# 実データ（camera）の準備

実データ（R `bayesm` パッケージの `camera` データセット）を long 形式 CSV に変換する。

camera データの仕様（`rdata` で `data/camera.RData` を parse して確認済み）:

- 332 人の回答者。各回答者は list の1要素で、辞書 `{"y": ..., "X": ...}` を持つ
- `y`: 長さ16のベクトル（1課題ごとの選択結果）。値は 1〜5（1〜4=提示された4案、5=どれも選ばない）
- `X`: (80, 10) の行列。80 = 16課題 × 5行（4案 + none 案）。
  列は `["canon","sony","nikon","panasonic","pixels","zoom","video","swivel","wifi","price"]`。
  ブランド4列は各行で必ずどれか1つだけが1（回答者に「ブランドなし」の選択肢はない）。
  pixels/zoom/video/swivel/wifi は 0/1 ダミー（1=高機能側）。
  price は100ドル単位の実数（0.79〜2.79）。none 行は全列0。
- 出典: https://github.com/cran/bayesm （`data/camera.RData`）
  仕様: https://rdrr.io/cran/bayesm/man/camera.html

出力する long CSV のスキーマは `prepare_persearch_camera.ipynb` の出力と揃える（詳細は README.md）。


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import rdata


In [ ]:
DATA_DIR = Path.cwd().parent / "data"
RESULTS_DIR = Path.cwd().parent / "results"

RDATA_PATH = DATA_DIR / "camera.RData"
OUTPUT_PATH = RESULTS_DIR / "camera_long.csv"

BRAND_COLS = ["canon", "sony", "nikon", "panasonic"]
FEATURE_COLS = ["pixels", "zoom", "video", "swivel", "wifi"]
FEATURE_LABELS = {
    "pixels": ("低画素", "高画素"),
    "zoom": ("標準ズーム", "高倍率ズーム"),
    "video": ("動画機能なし", "動画機能あり"),
    "swivel": ("回転式液晶なし", "回転式液晶あり"),
    "wifi": ("Wi-Fiなし", "Wi-Fiあり"),
}
OPTION_LABELS = ["A", "B", "C", "D"]  # none は別途 "none"
N_TASKS = 16
N_ALTS = 5  # A/B/C/D + none


## 関数定義

In [ ]:
def load_camera() -> list[dict]:
    parsed = rdata.parser.parse_file(str(RDATA_PATH))
    converted = rdata.conversion.convert(parsed)
    return converted["camera"]


In [ ]:
def build_long_rows(camera: list[dict]) -> list[dict]:
    rows: list[dict] = []
    for resp_idx, resp in enumerate(camera):
        respondent_id = resp_idx + 1
        y = np.asarray(resp["y"]).astype(int)  # 長さ16、1〜5
        X = np.asarray(resp["X"].values).astype(float)  # (80, 10)

        if X.shape != (N_TASKS * N_ALTS, 10):
            raise ValueError(f"respondent {respondent_id}: unexpected X shape {X.shape}")
        if len(y) != N_TASKS:
            raise ValueError(f"respondent {respondent_id}: unexpected y length {len(y)}")

        for task_index in range(N_TASKS):
            block = X[task_index * N_ALTS : (task_index + 1) * N_ALTS]
            chosen_option_number = int(y[task_index])  # 1〜5

            for alt_idx in range(N_ALTS):
                row_vals = block[alt_idx]
                is_none = alt_idx == N_ALTS - 1  # 5番目(index4)が none 行
                option_number = alt_idx + 1
                option_label = "none" if is_none else OPTION_LABELS[alt_idx]

                if is_none:
                    brand = None
                    features = {k: None for k in FEATURE_COLS}
                    price_usd100 = None
                else:
                    brand_vals = row_vals[:4]
                    if brand_vals.sum() != 1:
                        raise ValueError(
                            f"respondent {respondent_id} task {task_index} alt {alt_idx}: "
                            f"brand dummies do not sum to 1 ({brand_vals})"
                        )
                    brand = BRAND_COLS[int(np.argmax(brand_vals))]
                    features = {}
                    for i, col in enumerate(FEATURE_COLS):
                        flag = int(row_vals[4 + i])
                        features[col] = FEATURE_LABELS[col][flag]
                    price_usd100 = float(row_vals[9])

                rows.append(
                    {
                        "respondent_id": respondent_id,
                        "task_index": task_index,
                        "option_label": option_label,
                        "is_none": int(is_none),
                        "chosen": int(option_number == chosen_option_number),
                        "brand": brand,
                        **features,
                        "price_usd100": price_usd100,
                    }
                )
    return rows


In [ ]:
def validate(df: pd.DataFrame) -> None:
    n_resp = df["respondent_id"].nunique()
    assert n_resp == 332, f"respondent count mismatch: {n_resp}"
    assert len(df) == 332 * N_TASKS * N_ALTS, f"row count mismatch: {len(df)}"

    # 1課題 = 5行、chosen はちょうど1行
    chosen_per_task = df.groupby(["respondent_id", "task_index"])["chosen"].sum()
    assert (chosen_per_task == 1).all(), "each task must have exactly one chosen row"

    # none 行は課題ごとにちょうど1行
    none_per_task = df.groupby(["respondent_id", "task_index"])["is_none"].sum()
    assert (none_per_task == 1).all(), "each task must have exactly one none row"

    rows_per_task = df.groupby(["respondent_id", "task_index"]).size()
    assert (rows_per_task == N_ALTS).all(), "each task must have exactly 5 rows"


## 実行

`data/camera.RData` を README の手順で取得済みであること。

In [ ]:
camera = load_camera()
rows = build_long_rows(camera)
df = pd.DataFrame(rows)
validate(df)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"wrote {len(df)} rows to {OUTPUT_PATH}")
print(f"respondents: {df['respondent_id'].nunique()}, tasks/respondent: {N_TASKS}")
print("none choice rate:", df.loc[df['is_none'] == 1, 'chosen'].mean())
print("price levels (USD/100):", sorted(df['price_usd100'].dropna().unique()))
